<a href="https://colab.research.google.com/github/renatobdo/doutorado/blob/main/05_geracao_documentos_agregados_sim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Título e objetivo

# Geração de documentos semânticos agregados do SIM

Este notebook transforma os resultados agregados das análises dos óbitos por
dengue no Sistema de Informações sobre Mortalidade (SIM) em documentos
semanticamente estruturados.

Os documentos produzidos sintetizam:

1. o panorama geral dos óbitos por dengue;
2. as principais causas e condições associadas;
3. as condições registradas na Parte I da Declaração de Óbito;
4. as condições registradas na Parte II;
5. as condições mencionadas em cada campo da declaração;
6. o perfil dos óbitos classificados como A90;
7. o perfil dos óbitos classificados como A91;
8. a comparação entre A90 e A91;
9. o perfil demográfico dos óbitos.

Cada documento contém texto legível, indicadores estruturados, escopo,
proveniência e metadados para futura indexação e recuperação semântica.

2 - Importação das bibliotecas

In [83]:
import os
import json
import math
import pandas as pd


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


3. Definição dos caminhos

Os nomes abaixo seguem os arquivos produzidos nos notebooks 03 e 04.

In [84]:
BASE_DIR = (
    "/content/drive/MyDrive/Doutorado/"
    "arbovirus_rag"
)

CAMINHO_CAUSAS_ASSOCIADAS = os.path.join(
    BASE_DIR,
    "data_processed",
    "sim_obitos_dengue_2023_causas_associadas.csv"
)

CAMINHO_CAUSAS_POR_CAMPO = os.path.join(
    BASE_DIR,
    "data_processed",
    "sim_obitos_dengue_2023_causas_por_campo.csv"
)

CAMINHO_RESUMO_A90_A91 = os.path.join(
    BASE_DIR,
    "data_processed",
    "sim_obitos_dengue_2023_resumo_a90_a91.csv"
)

CAMINHO_COMPARACAO_A90_A91 = os.path.join(
    BASE_DIR,
    "data_processed",
    "sim_obitos_dengue_2023_comparacao_causas_a90_a91.csv"
)

CAMINHO_PERFIL_DEMOGRAFICO = os.path.join(
    BASE_DIR,
    "data_processed",
    "sim_obitos_dengue_2023_perfil_demografico_a90_a91.csv"
)

CAMINHO_CAUSAS_NORMALIZADAS = os.path.join(
    BASE_DIR,
    "data_processed",
    "sim_obitos_dengue_2023_causas_cid10.csv"
)
PASTA_DOCUMENTOS_SIM = os.path.join(
    BASE_DIR,
    "data_docs",
    "SIM",
    "2023"
)

Caminhos de saída:

In [85]:
CAMINHO_SAIDA_JSON = os.path.join(
    BASE_DIR,
    "data_docs",
    "SIM",
    "sim_obitos_dengue_2023_documentos_agregados.json"
)

CAMINHO_SAIDA_JSONL = os.path.join(
    BASE_DIR,
    "data_docs",
    "SIM",
    "sim_obitos_dengue_2023_documentos_agregados.jsonl"
)

CAMINHO_SAIDA_CSV = os.path.join(
    BASE_DIR,
    "data_docs",
    "SIM",
    "sim_obitos_dengue_2023_documentos_agregados.csv"
)

NOMES_ARQUIVOS = {
    "SIM_DENGUE_2023_PANORAMA_GERAL":
        "01_panorama_geral.json",

    "SIM_DENGUE_2023_CAUSAS_ASSOCIADAS":
        "02_causas_associadas.json",

    "SIM_DENGUE_2023_PARTE_I":
        "03_parte_i_declaracao_obito.json",

    "SIM_DENGUE_2023_PARTE_II":
        "04_parte_ii_declaracao_obito.json",

    "SIM_DENGUE_2023_CAUSAS_POR_CAMPO":
        "05_causas_por_campo.json",

    "SIM_DENGUE_2023_PERFIL_A90":
        "06_perfil_a90.json",

    "SIM_DENGUE_2023_PERFIL_A91":
        "07_perfil_a91.json",

    "SIM_DENGUE_2023_COMPARACAO_A90_A91":
        "08_comparacao_a90_a91.json",

    "SIM_DENGUE_2023_PERFIL_DEMOGRAFICO":
        "09_perfil_demografico.json"
}

4. Verificação dos arquivos de entrada

In [86]:
arquivos_obrigatorios = [
    CAMINHO_CAUSAS_ASSOCIADAS,
    CAMINHO_CAUSAS_POR_CAMPO,
    CAMINHO_RESUMO_A90_A91,
    CAMINHO_COMPARACAO_A90_A91,
    CAMINHO_PERFIL_DEMOGRAFICO,
    CAMINHO_CAUSAS_NORMALIZADAS
]

for caminho in arquivos_obrigatorios:
    if not os.path.exists(caminho):
        raise FileNotFoundError(
            f"Arquivo não encontrado: {caminho}"
        )

print("Todos os arquivos necessários foram encontrados.")

Todos os arquivos necessários foram encontrados.


5. Carregamento das tabelas

In [87]:
df_causas_associadas = pd.read_csv(
    CAMINHO_CAUSAS_ASSOCIADAS,
    encoding="utf-8-sig"
)

df_causas_por_campo = pd.read_csv(
    CAMINHO_CAUSAS_POR_CAMPO,
    encoding="utf-8-sig"
)

df_resumo_a90_a91 = pd.read_csv(
    CAMINHO_RESUMO_A90_A91,
    encoding="utf-8-sig"
)

df_comparacao_a90_a91 = pd.read_csv(
    CAMINHO_COMPARACAO_A90_A91,
    encoding="utf-8-sig"
)

df_perfil_demografico = pd.read_csv(
    CAMINHO_PERFIL_DEMOGRAFICO,
    encoding="utf-8-sig"
)

df_causas_normalizadas = pd.read_csv(
    CAMINHO_CAUSAS_NORMALIZADAS,
    encoding="utf-8-sig"
)

Resumo

In [88]:
print("Causas associadas:", df_causas_associadas.shape)
print("Causas por campo:", df_causas_por_campo.shape)
print("Resumo A90/A91:", df_resumo_a90_a91.shape)
print("Comparação A90/A91:", df_comparacao_a90_a91.shape)
print("Perfil demográfico:", df_perfil_demografico.shape)
print("Causas normalizadas:", df_causas_normalizadas.shape)

Causas associadas: (420, 5)
Causas por campo: (699, 6)
Resumo A90/A91: (2, 3)
Comparação A90/A91: (420, 11)
Perfil demográfico: (30, 5)
Causas normalizadas: (6266, 9)


6. Conversão dos tipos numéricos

Como os arquivos são CSV, é útil garantir que contagens e percentuais sejam numéricos.

In [89]:
def converter_colunas_numericas(
    dataframe,
    colunas
):
    """
    Converte as colunas indicadas para valores numéricos.
    Valores inválidos são convertidos para NaN.
    """
    df = dataframe.copy()

    for coluna in colunas:
        if coluna in df.columns:
            df[coluna] = pd.to_numeric(
                df[coluna],
                errors="coerce"
            )

    return df

Aplicação

In [90]:
df_causas_associadas = converter_colunas_numericas(
    df_causas_associadas,
    [
        "total_mencoes",
        "total_obitos",
        "percentual_obitos"
    ]
)

df_causas_por_campo = converter_colunas_numericas(
    df_causas_por_campo,
    [
        "total_mencoes",
        "total_obitos"
    ]
)

df_resumo_a90_a91 = converter_colunas_numericas(
    df_resumo_a90_a91,
    [
        "total_obitos",
        "percentual"
    ]
)

df_comparacao_a90_a91 = converter_colunas_numericas(
    df_comparacao_a90_a91,
    [
        "obitos_a90",
        "total_grupo_a90",
        "percentual_a90",
        "obitos_a91",
        "total_grupo_a91",
        "percentual_a91",
        "diferenca_pp_a91_menos_a90",
        "razao_prevalencias_a91_a90"
    ]
)

df_perfil_demografico = converter_colunas_numericas(
    df_perfil_demografico,
    [
        "TOTAL_OBITOS",
        "PERCENTUAL"
    ]
)

7. Funções auxiliares
7.1 Tratamento de valores ausentes

In [91]:
def valor_json(valor):
    """
    Converte valores do pandas para tipos compatíveis com JSON.
    """
    if pd.isna(valor):
        return None

    if isinstance(valor, bool):
        return valor

    if isinstance(valor, int):
        return int(valor)

    if isinstance(valor, float):
        if math.isnan(valor):
            return None

        if valor.is_integer():
            return int(valor)

        return round(float(valor), 4)

    return str(valor)

7.2 Conversão de DataFrame para lista de indicadores

In [92]:
def dataframe_para_registros(
    dataframe,
    colunas=None
):
    """
    Converte um DataFrame em uma lista de dicionários
    compatíveis com JSON.
    """
    df = dataframe.copy()

    if colunas is not None:
        colunas_existentes = [
            coluna
            for coluna in colunas
            if coluna in df.columns
        ]

        df = df[colunas_existentes]

    registros = []

    for _, linha in df.iterrows():
        registro = {
            coluna: valor_json(linha[coluna])
            for coluna in df.columns
        }

        registros.append(registro)

    return registros

7.3 Formatação de percentual

In [93]:
def formatar_percentual(valor):
    if pd.isna(valor):
        return "percentual não disponível"

    return f"{float(valor):.2f}%".replace(".", ",")

7.4 Formatação de contagens

In [94]:
def formatar_inteiro(valor):
    if pd.isna(valor):
        return "0"

    return f"{int(valor):,}".replace(",", ".")

7.5 Criação de um documento padrão

In [95]:
def criar_documento_agregado(
    document_id,
    titulo,
    tipo_documento,
    tema,
    texto,
    indicadores,
    escopo=None,
    palavras_chave=None
):
    """
    Cria a estrutura padronizada de um documento agregado.
    """
    if escopo is None:
        escopo = {}

    if palavras_chave is None:
        palavras_chave = []

    return {
        "document_id": document_id,
        "document_type": tipo_documento,
        "title": titulo,
        "theme": tema,
        "scope": {
            "source_system": "SIM",
            "source_system_name": (
                "Sistema de Informações sobre Mortalidade"
            ),
            "source_file": "DOBR2023.dbc",
            "year": 2023,
            "disease": "Dengue",
            "classification_system": "CID-10",
            **escopo
        },
        "keywords": palavras_chave,
        "indicators": indicadores,
        "text": texto,
        "provenance": {
            "data_origin": (
                "Sistema de Informações sobre Mortalidade"
            ),
            "processing": (
                "Registros filtrados por causa básica A90 ou A91, "
                "normalizados e associados às descrições da CID-10."
            ),
            "aggregation_notice": (
                "Os valores apresentados são agregados e não "
                "representam registros individuais."
            ),
            "internal_identifier_notice": (
                "ID_REGISTRO foi utilizado apenas durante o "
                "processamento e não corresponde a um identificador "
                "oficial do SIM."
            )
        }
    }

8. Cálculo do resumo geral

A tabela de causas normalizadas permite recuperar o total de óbitos e outras medidas.

In [96]:
total_obitos = (
    df_causas_normalizadas["ID_REGISTRO"]
    .nunique()
)

total_mencoes = len(
    df_causas_normalizadas
)

total_cids_distintos = (
    df_causas_normalizadas["CODIGO_SIM"]
    .nunique()
)

Para identificar os óbitos com condições associadas, exclua a causa básica e as próprias menções de dengue:

In [97]:
df_associadas = df_causas_normalizadas[
    (df_causas_normalizadas["CAMPO_CAUSA"] != "CAUSABAS")
    &
    (
        ~df_causas_normalizadas[
            "CODIGO_CATEGORIA"
        ].isin(["A90", "A91"])
    )
].copy()

In [98]:
total_obitos_com_associadas = (
    df_associadas["ID_REGISTRO"]
    .nunique()
)

total_obitos_sem_associadas = (
    total_obitos
    - total_obitos_com_associadas
)

total_cids_associados = (
    df_associadas["CODIGO_SIM"]
    .nunique()
)

total_mencoes_associadas = len(
    df_associadas
)

9. Documento 1 — Panorama geral

In [99]:
texto_panorama = (
    f"Em 2023, foram identificados "
    f"{formatar_inteiro(total_obitos)} óbitos no SIM cuja causa "
    f"básica foi classificada como dengue, utilizando os códigos "
    f"A90 ou A91 da CID-10. "
    f"Entre esses registros, "
    f"{formatar_inteiro(total_obitos_com_associadas)} óbitos "
    f"apresentaram pelo menos uma causa ou condição associada além "
    f"da própria dengue, enquanto "
    f"{formatar_inteiro(total_obitos_sem_associadas)} não "
    f"apresentaram outras condições após a aplicação dos critérios "
    f"de filtragem. Foram identificados "
    f"{formatar_inteiro(total_cids_associados)} códigos CID-10 "
    f"associados distintos e "
    f"{formatar_inteiro(total_mencoes_associadas)} menções de "
    f"condições associadas."
)

In [100]:
documento_panorama = criar_documento_agregado(
    document_id="SIM_DENGUE_2023_PANORAMA_GERAL",
    titulo="Panorama geral dos óbitos por dengue no SIM em 2023",
    tipo_documento="epidemiological_summary",
    tema="Panorama geral",
    texto=texto_panorama,
    indicadores=[
        {
            "indicator": "total_obitos_dengue",
            "value": total_obitos,
            "unit": "óbitos"
        },
        {
            "indicator": "obitos_com_causa_associada",
            "value": total_obitos_com_associadas,
            "unit": "óbitos"
        },
        {
            "indicator": "obitos_sem_causa_associada",
            "value": total_obitos_sem_associadas,
            "unit": "óbitos"
        },
        {
            "indicator": "codigos_associados_distintos",
            "value": total_cids_associados,
            "unit": "códigos CID-10"
        },
        {
            "indicator": "mencoes_associadas",
            "value": total_mencoes_associadas,
            "unit": "menções"
        }
    ],
    escopo={
        "basic_cause_categories": ["A90", "A91"]
    },
    palavras_chave=[
        "SIM",
        "dengue",
        "mortalidade",
        "óbitos",
        "CID-10",
        "A90",
        "A91"
    ]
)

10. Documento 2 — Principais condições associadas

Selecione, por exemplo, as 20 principais:

In [101]:
TOP_N = 20

top_causas_associadas = (
    df_causas_associadas
    .sort_values(
        "total_obitos",
        ascending=False
    )
    .head(TOP_N)
    .copy()
)

Construa frases:

In [102]:
frases_causas = []

for _, linha in top_causas_associadas.iterrows():
    codigo = linha["CODIGO_SIM"]
    descricao = linha["DESCRICAO_CID"]
    total = linha["total_obitos"]
    percentual = linha["percentual_obitos"]

    frases_causas.append(
        f"{codigo} — {descricao} foi mencionado em "
        f"{formatar_inteiro(total)} óbitos "
        f"({formatar_percentual(percentual)})"
    )

In [103]:
texto_causas_associadas = (
    f"Entre os {formatar_inteiro(total_obitos)} óbitos cuja causa "
    f"básica foi dengue em 2023, as condições associadas mais "
    f"frequentes foram: "
    + "; ".join(frases_causas)
    + ". Os percentuais foram calculados em relação ao total de "
    "óbitos por dengue analisados. Uma mesma declaração pode conter "
    "mais de uma condição associada, portanto os percentuais não "
    "devem ser somados como partes de um único total."
)

In [104]:
documento_causas_associadas = criar_documento_agregado(
    document_id="SIM_DENGUE_2023_CAUSAS_ASSOCIADAS",
    titulo="Principais condições associadas aos óbitos por dengue em 2023",
    tipo_documento="aggregated_clinical_evidence",
    tema="Causas associadas",
    texto=texto_causas_associadas,
    indicadores=dataframe_para_registros(
        top_causas_associadas,
        [
            "CODIGO_SIM",
            "DESCRICAO_CID",
            "total_mencoes",
            "total_obitos",
            "percentual_obitos"
        ]
    ),
    escopo={
        "total_deaths": total_obitos,
        "ranking_size": TOP_N,
        "excludes_basic_cause": True
    },
    palavras_chave=[
        "causas associadas",
        "condições associadas",
        "mortalidade por dengue",
        "CID-10",
        "SIM"
    ]
)

11. Preparação da Parte I e Parte II

Caso PARTE_DECLARACAO não esteja no arquivo, crie a coluna:

In [105]:
if "PARTE_DECLARACAO" not in df_causas_por_campo.columns:
    df_causas_por_campo["PARTE_DECLARACAO"] = (
        df_causas_por_campo["CAMPO_CAUSA"]
        .map({
            "LINHAA": "Parte I",
            "LINHAB": "Parte I",
            "LINHAC": "Parte I",
            "LINHAD": "Parte I",
            "LINHAII": "Parte II"
        })
    )

Agregue novamente para evitar que um mesmo CID apareça fragmentado entre as linhas A, B, C e D:

In [106]:
frequencia_por_parte = (
    df_causas_por_campo
    .groupby(
        [
            "PARTE_DECLARACAO",
            "CODIGO_SIM",
            "DESCRICAO_CID"
        ],
        dropna=False
    )
    .agg(
        total_mencoes=("total_mencoes", "sum"),
        total_obitos=("total_obitos", "sum")
    )
    .reset_index()
)

Essa soma de total_obitos pode superestimar o número de óbitos caso o mesmo CID apareça em mais de uma linha do mesmo registro. O ideal é que o notebook 03 também salve uma tabela calculada diretamente por ID_REGISTRO e parte da declaração.

Para o documento textual, utilizaremos a tabela como frequência de registros por campo. Deixe essa limitação explícita nos metadados.

12. Documento 3 — Parte I

In [107]:
top_parte_i = (
    frequencia_por_parte[
        frequencia_por_parte[
            "PARTE_DECLARACAO"
        ] == "Parte I"
    ]
    .sort_values(
        "total_obitos",
        ascending=False
    )
    .head(TOP_N)
    .copy()
)

In [108]:
frases_parte_i = []

for _, linha in top_parte_i.iterrows():
    frases_parte_i.append(
        f"{linha['CODIGO_SIM']} — "
        f"{linha['DESCRICAO_CID']} "
        f"({formatar_inteiro(linha['total_obitos'])} registros)"
    )

In [109]:
texto_parte_i = (
    "A Parte I da Declaração de Óbito registra a sequência de "
    "condições que levou diretamente à morte. Nos óbitos cuja causa "
    "básica foi dengue em 2023, as condições mais frequentes nessa "
    "parte foram: "
    + "; ".join(frases_parte_i)
    + ". A posição na Parte I indica que essas condições foram "
    "registradas na sequência causal do óbito, mas não permite, "
    "isoladamente, estabelecer relações causais além das informações "
    "declaradas no documento."
)

In [110]:
documento_parte_i = criar_documento_agregado(
    document_id="SIM_DENGUE_2023_PARTE_I",
    titulo="Condições registradas na Parte I dos óbitos por dengue",
    tipo_documento="death_certificate_part_summary",
    tema="Parte I da Declaração de Óbito",
    texto=texto_parte_i,
    indicadores=dataframe_para_registros(
        top_parte_i,
        [
            "CODIGO_SIM",
            "DESCRICAO_CID",
            "total_mencoes",
            "total_obitos"
        ]
    ),
    escopo={
        "death_certificate_part": "Parte I",
        "meaning": "Sequência causal registrada"
    },
    palavras_chave=[
        "Parte I",
        "sequência causal",
        "Declaração de Óbito",
        "dengue",
        "SIM"
    ]
)

13. Documento 4 — Parte II

In [111]:
top_parte_ii = (
    frequencia_por_parte[
        frequencia_por_parte[
            "PARTE_DECLARACAO"
        ] == "Parte II"
    ]
    .sort_values(
        "total_obitos",
        ascending=False
    )
    .head(TOP_N)
    .copy()
)

In [112]:
frases_parte_ii = []

for _, linha in top_parte_ii.iterrows():
    frases_parte_ii.append(
        f"{linha['CODIGO_SIM']} — "
        f"{linha['DESCRICAO_CID']} "
        f"({formatar_inteiro(linha['total_obitos'])} registros)"
    )

In [113]:
texto_parte_ii = (
    "A Parte II da Declaração de Óbito reúne condições que "
    "contribuíram para a morte, mas que não foram registradas como "
    "parte direta da sequência causal apresentada na Parte I. "
    "Nos óbitos por dengue analisados, as condições mais frequentes "
    "na Parte II foram: "
    + "; ".join(frases_parte_ii)
    + ". A presença de uma condição na Parte II não significa "
    "automaticamente que ela seja uma comorbidade, mas indica que foi "
    "considerada uma condição contribuinte no preenchimento da "
    "declaração."
)

In [114]:
documento_parte_ii = criar_documento_agregado(
    document_id="SIM_DENGUE_2023_PARTE_II",
    titulo="Condições registradas na Parte II dos óbitos por dengue",
    tipo_documento="death_certificate_part_summary",
    tema="Parte II da Declaração de Óbito",
    texto=texto_parte_ii,
    indicadores=dataframe_para_registros(
        top_parte_ii,
        [
            "CODIGO_SIM",
            "DESCRICAO_CID",
            "total_mencoes",
            "total_obitos"
        ]
    ),
    escopo={
        "death_certificate_part": "Parte II",
        "meaning": "Condições contribuintes registradas"
    },
    palavras_chave=[
        "Parte II",
        "condições contribuintes",
        "comorbidades",
        "Declaração de Óbito",
        "dengue"
    ]
)

14. Documento 5 — Principais condições por campo

Selecione as cinco principais de cada linha:

In [115]:
TOP_POR_CAMPO = 5

top_por_campo = (
    df_causas_por_campo
    .sort_values(
        [
            "CAMPO_CAUSA",
            "total_obitos"
        ],
        ascending=[True, False]
    )
    .groupby(
        "CAMPO_CAUSA",
        group_keys=False
    )
    .head(TOP_POR_CAMPO)
    .copy()
)

Construa o texto por campo:

In [116]:
descricoes_campos = {
    "LINHAA": "Linha A, geralmente mais próxima da causa imediata",
    "LINHAB": "Linha B da sequência causal",
    "LINHAC": "Linha C da sequência causal",
    "LINHAD": "Linha D da sequência causal",
    "LINHAII": "Parte II, destinada às condições contribuintes"
}

In [117]:
paragrafos_campos = []

for campo, grupo in top_por_campo.groupby(
    "CAMPO_CAUSA"
):
    condicoes = []

    for _, linha in grupo.iterrows():
        condicoes.append(
            f"{linha['CODIGO_SIM']} — "
            f"{linha['DESCRICAO_CID']} "
            f"({formatar_inteiro(linha['total_obitos'])})"
        )

    descricao_campo = descricoes_campos.get(
        campo,
        campo
    )

    paragrafos_campos.append(
        f"{descricao_campo}: "
        + "; ".join(condicoes)
        + "."
    )

In [118]:
texto_por_campo = (
    "A posição de uma condição na Declaração de Óbito ajuda a "
    "interpretar o papel atribuído a ela no preenchimento do "
    "documento. "
    + " ".join(paragrafos_campos)
    + " As frequências por campo devem ser interpretadas separadamente, "
    "pois uma mesma condição pode aparecer em posições diferentes."
)

In [119]:
documento_por_campo = criar_documento_agregado(
    document_id="SIM_DENGUE_2023_CAUSAS_POR_CAMPO",
    titulo="Principais condições por posição na Declaração de Óbito",
    tipo_documento="death_certificate_position_summary",
    tema="Posição das causas na Declaração de Óbito",
    texto=texto_por_campo,
    indicadores=dataframe_para_registros(
        top_por_campo,
        [
            "CAMPO_CAUSA",
            "PAPEL_CAUSA",
            "CODIGO_SIM",
            "DESCRICAO_CID",
            "total_mencoes",
            "total_obitos"
        ]
    ),
    escopo={
        "fields": [
            "LINHAA",
            "LINHAB",
            "LINHAC",
            "LINHAD",
            "LINHAII"
        ]
    },
    palavras_chave=[
        "Linha A",
        "Linha B",
        "Linha C",
        "Linha D",
        "Parte II",
        "cadeia causal"
    ]
)

15. Documento 6 — Perfil de A90

Primeiro recupere o total de A90:

In [120]:
linha_a90 = df_resumo_a90_a91[
    df_resumo_a90_a91["TIPO_DENGUE"]
    == "Dengue (A90)"
]

if linha_a90.empty:
    raise ValueError(
        "O grupo Dengue (A90) não foi encontrado."
    )

total_a90 = int(
    linha_a90["total_obitos"].iloc[0]
)

percentual_total_a90 = float(
    linha_a90["percentual"].iloc[0]
)

Selecione as principais condições:

In [121]:
top_a90 = (
    df_comparacao_a90_a91[
        df_comparacao_a90_a91["obitos_a90"] > 0
    ]
    .sort_values(
        "percentual_a90",
        ascending=False
    )
    .head(TOP_N)
    .copy()
)

In [122]:
frases_a90 = []

for _, linha in top_a90.iterrows():
    frases_a90.append(
        f"{linha['CODIGO_SIM']} — "
        f"{linha['DESCRICAO_CID']} foi mencionado em "
        f"{formatar_inteiro(linha['obitos_a90'])} óbitos "
        f"({formatar_percentual(linha['percentual_a90'])})"
    )

In [123]:
texto_a90 = (
    f"O grupo classificado como dengue A90 reuniu "
    f"{formatar_inteiro(total_a90)} óbitos, correspondendo a "
    f"{formatar_percentual(percentual_total_a90)} dos óbitos por "
    f"dengue analisados. As condições associadas mais frequentes "
    f"nesse grupo foram: "
    + "; ".join(frases_a90)
    + ". Os percentuais foram calculados dentro do próprio grupo A90."
)

In [124]:
documento_a90 = criar_documento_agregado(
    document_id="SIM_DENGUE_2023_PERFIL_A90",
    titulo="Perfil das condições associadas aos óbitos classificados como A90",
    tipo_documento="disease_subgroup_summary",
    tema="Dengue A90",
    texto=texto_a90,
    indicadores=dataframe_para_registros(
        top_a90,
        [
            "CODIGO_SIM",
            "DESCRICAO_CID",
            "obitos_a90",
            "percentual_a90"
        ]
    ),
    escopo={
        "basic_cause_category": "A90",
        "total_deaths": total_a90
    },
    palavras_chave=[
        "A90",
        "dengue",
        "causa básica",
        "condições associadas"
    ]
)

16. Documento 7 — Perfil de A91

In [125]:
linha_a91 = df_resumo_a90_a91[
    df_resumo_a90_a91["TIPO_DENGUE"]
    == "Dengue hemorrágica (A91)"
]

if linha_a91.empty:
    raise ValueError(
        "O grupo Dengue hemorrágica (A91) não foi encontrado."
    )

total_a91 = int(
    linha_a91["total_obitos"].iloc[0]
)

percentual_total_a91 = float(
    linha_a91["percentual"].iloc[0]
)

In [126]:
top_a91 = (
    df_comparacao_a90_a91[
        df_comparacao_a90_a91["obitos_a91"] > 0
    ]
    .sort_values(
        "percentual_a91",
        ascending=False
    )
    .head(TOP_N)
    .copy()
)

In [127]:
frases_a91 = []

for _, linha in top_a91.iterrows():
    frases_a91.append(
        f"{linha['CODIGO_SIM']} — "
        f"{linha['DESCRICAO_CID']} foi mencionado em "
        f"{formatar_inteiro(linha['obitos_a91'])} óbitos "
        f"({formatar_percentual(linha['percentual_a91'])})"
    )

In [128]:
texto_a91 = (
    f"O grupo classificado como dengue hemorrágica A91 reuniu "
    f"{formatar_inteiro(total_a91)} óbitos, correspondendo a "
    f"{formatar_percentual(percentual_total_a91)} dos óbitos por "
    f"dengue analisados. As condições associadas mais frequentes "
    f"nesse grupo foram: "
    + "; ".join(frases_a91)
    + ". Os percentuais foram calculados dentro do próprio grupo A91."
)

In [129]:
documento_a91 = criar_documento_agregado(
    document_id="SIM_DENGUE_2023_PERFIL_A91",
    titulo="Perfil das condições associadas aos óbitos classificados como A91",
    tipo_documento="disease_subgroup_summary",
    tema="Dengue hemorrágica A91",
    texto=texto_a91,
    indicadores=dataframe_para_registros(
        top_a91,
        [
            "CODIGO_SIM",
            "DESCRICAO_CID",
            "obitos_a91",
            "percentual_a91"
        ]
    ),
    escopo={
        "basic_cause_category": "A91",
        "total_deaths": total_a91
    },
    palavras_chave=[
        "A91",
        "dengue hemorrágica",
        "causa básica",
        "condições associadas"
    ]
)

17. Documento 8 — Comparação A90 versus A91

Use um mínimo de ocorrências para evitar destacar diferenças baseadas em um ou dois registros.

In [130]:
MINIMO_OBITOS_COMPARACAO = 5

df_comparacao_relevante = (
    df_comparacao_a90_a91[
        (
            df_comparacao_a90_a91["obitos_a90"]
            >= MINIMO_OBITOS_COMPARACAO
        )
        |
        (
            df_comparacao_a90_a91["obitos_a91"]
            >= MINIMO_OBITOS_COMPARACAO
        )
    ]
    .copy()
)

Principais diferenças favoráveis a A91:

In [131]:
top_diferencas_a91 = (
    df_comparacao_relevante
    .sort_values(
        "diferenca_pp_a91_menos_a90",
        ascending=False
    )
    .head(10)
    .copy()
)

Principais diferenças favoráveis a A90:

In [132]:
top_diferencas_a90 = (
    df_comparacao_relevante
    .sort_values(
        "diferenca_pp_a91_menos_a90",
        ascending=True
    )
    .head(10)
    .copy()
)

Construa os textos:

In [133]:
frases_diferencas_a91 = []

for _, linha in top_diferencas_a91.iterrows():
    frases_diferencas_a91.append(
        f"{linha['CODIGO_SIM']} — "
        f"{linha['DESCRICAO_CID']}: "
        f"{formatar_percentual(linha['percentual_a90'])} em A90 e "
        f"{formatar_percentual(linha['percentual_a91'])} em A91, "
        f"diferença de "
        f"{linha['diferenca_pp_a91_menos_a90']:.2f} pontos percentuais"
        .replace(".", ",")
    )

In [134]:
frases_diferencas_a90 = []

for _, linha in top_diferencas_a90.iterrows():
    diferenca_a90 = (
        linha["percentual_a90"]
        - linha["percentual_a91"]
    )

    frases_diferencas_a90.append(
        f"{linha['CODIGO_SIM']} — "
        f"{linha['DESCRICAO_CID']}: "
        f"{formatar_percentual(linha['percentual_a90'])} em A90 e "
        f"{formatar_percentual(linha['percentual_a91'])} em A91, "
        f"diferença de "
        f"{diferenca_a90:.2f} pontos percentuais em favor de A90"
        .replace(".", ",")
    )

In [135]:
texto_comparacao = (
    "A comparação entre os grupos A90 e A91 foi realizada utilizando "
    "o percentual de óbitos de cada grupo em que cada condição foi "
    "mencionada. As condições com maiores diferenças proporcionais "
    "em favor de A91 foram: "
    + "; ".join(frases_diferencas_a91)
    + ". As condições com maiores diferenças em favor de A90 foram: "
    + "; ".join(frases_diferencas_a90)
    + ". As diferenças apresentadas são descritivas e não representam, "
    "por si só, evidência de associação causal ou significância "
    "estatística."
)

Indicadores combinados

In [136]:
indicadores_comparacao = []

for _, linha in top_diferencas_a91.iterrows():
    indicador = {
        "direction": "Mais frequente proporcionalmente em A91",
        **{
            coluna: valor_json(linha[coluna])
            for coluna in [
                "CODIGO_SIM",
                "DESCRICAO_CID",
                "obitos_a90",
                "percentual_a90",
                "obitos_a91",
                "percentual_a91",
                "diferenca_pp_a91_menos_a90",
                "razao_prevalencias_a91_a90"
            ]
        }
    }

    indicadores_comparacao.append(indicador)

for _, linha in top_diferencas_a90.iterrows():
    indicador = {
        "direction": "Mais frequente proporcionalmente em A90",
        **{
            coluna: valor_json(linha[coluna])
            for coluna in [
                "CODIGO_SIM",
                "DESCRICAO_CID",
                "obitos_a90",
                "percentual_a90",
                "obitos_a91",
                "percentual_a91",
                "diferenca_pp_a91_menos_a90",
                "razao_prevalencias_a91_a90"
            ]
        }
    }

    indicadores_comparacao.append(indicador)

In [137]:
documento_comparacao = criar_documento_agregado(
    document_id="SIM_DENGUE_2023_COMPARACAO_A90_A91",
    titulo="Comparação das condições associadas entre A90 e A91",
    tipo_documento="comparative_epidemiological_summary",
    tema="Comparação A90 versus A91",
    texto=texto_comparacao,
    indicadores=indicadores_comparacao,
    escopo={
        "group_a": {
            "code": "A90",
            "total_deaths": total_a90
        },
        "group_b": {
            "code": "A91",
            "total_deaths": total_a91
        },
        "minimum_occurrences": MINIMO_OBITOS_COMPARACAO
    },
    palavras_chave=[
        "A90",
        "A91",
        "comparação",
        "diferença percentual",
        "razão de prevalências"
    ]
)

18. Documento 9 — Perfil demográfico

Organize o perfil por variável:

In [138]:
partes_perfil = []

for variavel, grupo in df_perfil_demografico.groupby(
    "VARIAVEL"
):
    descricoes = []

    grupo = grupo.sort_values(
        [
            "TIPO_DENGUE",
            "PERCENTUAL"
        ],
        ascending=[True, False]
    )

    for _, linha in grupo.iterrows():
        descricoes.append(
            f"{linha['TIPO_DENGUE']}, "
            f"{linha['CATEGORIA']}: "
            f"{formatar_inteiro(linha['TOTAL_OBITOS'])} óbitos "
            f"({formatar_percentual(linha['PERCENTUAL'])})"
        )

    partes_perfil.append(
        f"{variavel}: "
        + "; ".join(descricoes)
        + "."
    )

In [139]:
texto_perfil_demografico = (
    "O perfil demográfico dos óbitos por dengue foi analisado "
    "separadamente para os grupos A90 e A91. "
    + " ".join(partes_perfil)
    + " Os percentuais foram calculados dentro de cada grupo de causa "
    "básica e devem ser interpretados como distribuições descritivas."
)

In [140]:
documento_perfil_demografico = criar_documento_agregado(
    document_id="SIM_DENGUE_2023_PERFIL_DEMOGRAFICO",
    titulo="Perfil demográfico dos óbitos por dengue A90 e A91",
    tipo_documento="demographic_summary",
    tema="Perfil demográfico",
    texto=texto_perfil_demografico,
    indicadores=dataframe_para_registros(
        df_perfil_demografico,
        [
            "TIPO_DENGUE",
            "VARIAVEL",
            "CATEGORIA",
            "TOTAL_OBITOS",
            "PERCENTUAL"
        ]
    ),
    escopo={
        "groups": ["A90", "A91"],
        "variables": (
            df_perfil_demografico[
                "VARIAVEL"
            ]
            .dropna()
            .unique()
            .tolist()
        )
    },
    palavras_chave=[
        "sexo",
        "idade",
        "faixa etária",
        "raça/cor",
        "perfil demográfico",
        "mortalidade"
    ]
)

19. Reunir todos os documentos

In [141]:
documentos_agregados = [
    documento_panorama,
    documento_causas_associadas,
    documento_parte_i,
    documento_parte_ii,
    documento_por_campo,
    documento_a90,
    documento_a91,
    documento_comparacao,
    documento_perfil_demografico
]

print(
    "Documentos agregados gerados:",
    len(documentos_agregados)
)

Documentos agregados gerados: 9


O esperado é:

9

20. Visualização dos documentos

Veja os títulos:

In [142]:
for documento in documentos_agregados:
    print(
        documento["document_id"],
        "—",
        documento["title"]
    )

SIM_DENGUE_2023_PANORAMA_GERAL — Panorama geral dos óbitos por dengue no SIM em 2023
SIM_DENGUE_2023_CAUSAS_ASSOCIADAS — Principais condições associadas aos óbitos por dengue em 2023
SIM_DENGUE_2023_PARTE_I — Condições registradas na Parte I dos óbitos por dengue
SIM_DENGUE_2023_PARTE_II — Condições registradas na Parte II dos óbitos por dengue
SIM_DENGUE_2023_CAUSAS_POR_CAMPO — Principais condições por posição na Declaração de Óbito
SIM_DENGUE_2023_PERFIL_A90 — Perfil das condições associadas aos óbitos classificados como A90
SIM_DENGUE_2023_PERFIL_A91 — Perfil das condições associadas aos óbitos classificados como A91
SIM_DENGUE_2023_COMPARACAO_A90_A91 — Comparação das condições associadas entre A90 e A91
SIM_DENGUE_2023_PERFIL_DEMOGRAFICO — Perfil demográfico dos óbitos por dengue A90 e A91


Visualize um documento completo:

In [143]:
print(
    json.dumps(
        documentos_agregados[0],
        ensure_ascii=False,
        indent=2
    )
)

{
  "document_id": "SIM_DENGUE_2023_PANORAMA_GERAL",
  "document_type": "epidemiological_summary",
  "title": "Panorama geral dos óbitos por dengue no SIM em 2023",
  "theme": "Panorama geral",
  "scope": {
    "source_system": "SIM",
    "source_system_name": "Sistema de Informações sobre Mortalidade",
    "source_file": "DOBR2023.dbc",
    "year": 2023,
    "disease": "Dengue",
    "classification_system": "CID-10",
    "basic_cause_categories": [
      "A90",
      "A91"
    ]
  },
  "keywords": [
    "SIM",
    "dengue",
    "mortalidade",
    "óbitos",
    "CID-10",
    "A90",
    "A91"
  ],
  "indicators": [
    {
      "indicator": "total_obitos_dengue",
      "value": 1322,
      "unit": "óbitos"
    },
    {
      "indicator": "obitos_com_causa_associada",
      "value": 1254,
      "unit": "óbitos"
    },
    {
      "indicator": "obitos_sem_causa_associada",
      "value": 68,
      "unit": "óbitos"
    },
    {
      "indicator": "codigos_associados_distintos",
      "value

Veja apenas os textos:

In [144]:
for documento in documentos_agregados:
    print("=" * 80)
    print(documento["title"])
    print("=" * 80)
    print(documento["text"])
    print()

Panorama geral dos óbitos por dengue no SIM em 2023
Em 2023, foram identificados 1.322 óbitos no SIM cuja causa básica foi classificada como dengue, utilizando os códigos A90 ou A91 da CID-10. Entre esses registros, 1.254 óbitos apresentaram pelo menos uma causa ou condição associada além da própria dengue, enquanto 68 não apresentaram outras condições após a aplicação dos critérios de filtragem. Foram identificados 420 códigos CID-10 associados distintos e 3.585 menções de condições associadas.

Principais condições associadas aos óbitos por dengue em 2023
Entre os 1.322 óbitos cuja causa básica foi dengue em 2023, as condições associadas mais frequentes foram: A419 — Septicemia não especificada foi mencionado em 330 óbitos (24,96%); I10X — Hipertensão essencial (primária) foi mencionado em 247 óbitos (18,68%); R571 — Choque hipovolêmico foi mencionado em 189 óbitos (14,30%); N179 — Insuficiência renal aguda não especificada foi mencionado em 154 óbitos (11,65%); J189 — Pneumonia não 

21. Validação dos documentos

Identificadores únicos

In [145]:
document_ids = [
    documento["document_id"]
    for documento in documentos_agregados
]

if len(document_ids) != len(set(document_ids)):
    raise ValueError(
        "Foram encontrados document_id duplicados."
    )

print("Todos os document_id são únicos.")

Todos os document_id são únicos.


Textos não vazios

In [146]:
documentos_sem_texto = [
    documento["document_id"]
    for documento in documentos_agregados
    if not documento.get("text")
]

print(
    "Documentos sem texto:",
    documentos_sem_texto
)

Documentos sem texto: []


Indicadores não vazios

In [147]:
documentos_sem_indicadores = [
    documento["document_id"]
    for documento in documentos_agregados
    if not documento.get("indicators")
]

print(
    "Documentos sem indicadores:",
    documentos_sem_indicadores
)

Documentos sem indicadores: []


22. Salvamento em JSON

In [148]:
with open(
    CAMINHO_SAIDA_JSON,
    "w",
    encoding="utf-8"
) as arquivo:
    json.dump(
        documentos_agregados,
        arquivo,
        ensure_ascii=False,
        indent=2
    )

print(
    "Arquivo JSON salvo em:\n"
    f"{CAMINHO_SAIDA_JSON}"
)

Arquivo JSON salvo em:
/content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SIM/sim_obitos_dengue_2023_documentos_agregados.json


23. Salvamento em JSONL

No JSONL, cada linha será um documento independente.

In [149]:
with open(
    CAMINHO_SAIDA_JSONL,
    "w",
    encoding="utf-8"
) as arquivo:

    for documento in documentos_agregados:
        arquivo.write(
            json.dumps(
                documento,
                ensure_ascii=False
            )
            + "\n"
        )

print(
    "Arquivo JSONL salvo em:\n"
    f"{CAMINHO_SAIDA_JSONL}"
)

Arquivo JSONL salvo em:
/content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SIM/sim_obitos_dengue_2023_documentos_agregados.jsonl


24. Criação de uma versão tabular

Para indexação vetorial, pode ser útil manter uma versão simplificada:

In [150]:
registros_documentos = []

for documento in documentos_agregados:
    registros_documentos.append(
        {
            "DOCUMENT_ID": documento["document_id"],
            "DOCUMENT_TYPE": documento["document_type"],
            "TITLE": documento["title"],
            "THEME": documento["theme"],
            "YEAR": documento["scope"]["year"],
            "SOURCE_SYSTEM": documento[
                "scope"
            ]["source_system"],
            "KEYWORDS": "; ".join(
                documento["keywords"]
            ),
            "TOTAL_INDICATORS": len(
                documento["indicators"]
            ),
            "TEXT": documento["text"]
        }
    )

df_documentos_agregados = pd.DataFrame(
    registros_documentos
)

Visualização:

In [152]:
df_documentos_agregados[
    [
        "DOCUMENT_ID",
        "TITLE",
        "THEME",
        "TOTAL_INDICATORS"
    ]
]

,DOCUMENT_ID,TITLE,THEME,TOTAL_INDICATORS
0,SIM_DENGUE_2023_PANORAMA_GERAL,Panorama geral dos óbitos por dengue no SIM em...,Panorama geral,5
1,SIM_DENGUE_2023_CAUSAS_ASSOCIADAS,Principais condições associadas aos óbitos por...,Causas associadas,20
2,SIM_DENGUE_2023_PARTE_I,Condições registradas na Parte I dos óbitos po...,Parte I da Declaração de Óbito,20
3,SIM_DENGUE_2023_PARTE_II,Condições registradas na Parte II dos óbitos p...,Parte II da Declaração de Óbito,20
4,SIM_DENGUE_2023_CAUSAS_POR_CAMPO,Principais condições por posição na Declaração...,Posição das causas na Declaração de Óbito,25
5,SIM_DENGUE_2023_PERFIL_A90,Perfil das condições associadas aos óbitos cla...,Dengue A90,20
6,SIM_DENGUE_2023_PERFIL_A91,Perfil das condições associadas aos óbitos cla...,Dengue hemorrágica A91,20
7,SIM_DENGUE_2023_COMPARACAO_A90_A91,Comparação das condições associadas entre A90 ...,Comparação A90 versus A91,20
8,SIM_DENGUE_2023_PERFIL_DEMOGRAFICO,Perfil demográfico dos óbitos por dengue A90 e...,Perfil demográfico,30


25. Salvamento em CSV

In [153]:
df_documentos_agregados.to_csv(
    CAMINHO_SAIDA_CSV,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Arquivo CSV salvo em:\n"
    f"{CAMINHO_SAIDA_CSV}"
)

Arquivo CSV salvo em:
/content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SIM/sim_obitos_dengue_2023_documentos_agregados.csv


26. Teste de leitura do JSONL

In [154]:
documentos_teste = []

with open(
    CAMINHO_SAIDA_JSONL,
    "r",
    encoding="utf-8"
) as arquivo:

    for linha in arquivo:
        documentos_teste.append(
            json.loads(linha)
        )

print(
    "Documentos lidos do JSONL:",
    len(documentos_teste)
)

Documentos lidos do JSONL: 9


In [155]:
print(
    documentos_teste[0]["title"]
)

print(
    documentos_teste[0]["text"]
)

Panorama geral dos óbitos por dengue no SIM em 2023
Em 2023, foram identificados 1.322 óbitos no SIM cuja causa básica foi classificada como dengue, utilizando os códigos A90 ou A91 da CID-10. Entre esses registros, 1.254 óbitos apresentaram pelo menos uma causa ou condição associada além da própria dengue, enquanto 68 não apresentaram outras condições após a aplicação dos critérios de filtragem. Foram identificados 420 códigos CID-10 associados distintos e 3.585 menções de condições associadas.


Salvar cada documento separadamente

In [156]:
arquivos_gerados = []

for documento in documentos_agregados:

    document_id = documento["document_id"]

    nome_arquivo = NOMES_ARQUIVOS.get(
        document_id
    )

    if nome_arquivo is None:
        raise KeyError(
            f"Nome de arquivo não definido para: "
            f"{document_id}"
        )

    caminho_arquivo = os.path.join(
        PASTA_DOCUMENTOS_SIM,
        nome_arquivo
    )

    with open(
        caminho_arquivo,
        "w",
        encoding="utf-8"
    ) as arquivo:
        json.dump(
            documento,
            arquivo,
            ensure_ascii=False,
            indent=2
        )

    arquivos_gerados.append(
        {
            "document_id": document_id,
            "nome_arquivo": nome_arquivo,
            "caminho": caminho_arquivo
        }
    )

print(
    f"{len(arquivos_gerados)} documentos "
    f"salvos separadamente."
)

9 documentos salvos separadamente.


Conferir os arquivos gerados

In [157]:
df_arquivos_gerados = pd.DataFrame(
    arquivos_gerados
)

df_arquivos_gerados

,document_id,nome_arquivo,caminho
0,SIM_DENGUE_2023_PANORAMA_GERAL,01_panorama_geral.json,/content/drive/MyDrive/Doutorado/arbovirus_rag...
1,SIM_DENGUE_2023_CAUSAS_ASSOCIADAS,02_causas_associadas.json,/content/drive/MyDrive/Doutorado/arbovirus_rag...
2,SIM_DENGUE_2023_PARTE_I,03_parte_i_declaracao_obito.json,/content/drive/MyDrive/Doutorado/arbovirus_rag...
3,SIM_DENGUE_2023_PARTE_II,04_parte_ii_declaracao_obito.json,/content/drive/MyDrive/Doutorado/arbovirus_rag...
4,SIM_DENGUE_2023_CAUSAS_POR_CAMPO,05_causas_por_campo.json,/content/drive/MyDrive/Doutorado/arbovirus_rag...
5,SIM_DENGUE_2023_PERFIL_A90,06_perfil_a90.json,/content/drive/MyDrive/Doutorado/arbovirus_rag...
6,SIM_DENGUE_2023_PERFIL_A91,07_perfil_a91.json,/content/drive/MyDrive/Doutorado/arbovirus_rag...
7,SIM_DENGUE_2023_COMPARACAO_A90_A91,08_comparacao_a90_a91.json,/content/drive/MyDrive/Doutorado/arbovirus_rag...
8,SIM_DENGUE_2023_PERFIL_DEMOGRAFICO,09_perfil_demografico.json,/content/drive/MyDrive/Doutorado/arbovirus_rag...


Também pode listar diretamente a pasta:

In [158]:
for nome in sorted(
    os.listdir(PASTA_DOCUMENTOS_SIM)
):
    print(nome)

01_panorama_geral.json
02_causas_associadas.json
03_parte_i_declaracao_obito.json
04_parte_ii_declaracao_obito.json
05_causas_por_campo.json
06_perfil_a90.json
07_perfil_a91.json
08_comparacao_a90_a91.json
09_perfil_demografico.json


27. Resumo final

In [159]:
print("=" * 70)
print("RESUMO DA GERAÇÃO DE DOCUMENTOS AGREGADOS")
print("=" * 70)

print(
    f"Total de documentos gerados: "
    f"{len(documentos_agregados)}"
)

print(
    f"Total de óbitos representados: "
    f"{formatar_inteiro(total_obitos)}"
)

print(
    f"Documentos sobre causas associadas: 1"
)

print(
    f"Documentos sobre a Declaração de Óbito: 3"
)

print(
    f"Documentos sobre A90 e A91: 3"
)

print(
    f"Documentos demográficos: 1"
)

print("\nArquivos gerados:")

print(CAMINHO_SAIDA_JSON)
print(CAMINHO_SAIDA_JSONL)
print(CAMINHO_SAIDA_CSV)

RESUMO DA GERAÇÃO DE DOCUMENTOS AGREGADOS
Total de documentos gerados: 9
Total de óbitos representados: 1.322
Documentos sobre causas associadas: 1
Documentos sobre a Declaração de Óbito: 3
Documentos sobre A90 e A91: 3
Documentos demográficos: 1

Arquivos gerados:
/content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SIM/sim_obitos_dengue_2023_documentos_agregados.json
/content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SIM/sim_obitos_dengue_2023_documentos_agregados.jsonl
/content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SIM/sim_obitos_dengue_2023_documentos_agregados.csv


Estrutura dos documentos produzidos

O notebook produzirá documentos semelhantes a:

{
  "document_id": "SIM_DENGUE_2023_CAUSAS_ASSOCIADAS",
  "document_type": "aggregated_clinical_evidence",
  "title": "Principais condições associadas aos óbitos por dengue em 2023",
  "theme": "Causas associadas",
  "scope": {
    "source_system": "SIM",
    "year": 2023,
    "disease": "Dengue",
    "total_deaths": 1322
  },
  "keywords": [
    "causas associadas",
    "mortalidade por dengue",
    "CID-10"
  ],
  "indicators": [
    {
      "CODIGO_SIM": "A419",
      "DESCRICAO_CID": "Septicemia não especificada",
      "total_obitos": 330,
      "percentual_obitos": 24.96
    }
  ],
  "text": "Entre os 1.322 óbitos cuja causa básica foi dengue...",
  "provenance": {
    "data_origin": "Sistema de Informações sobre Mortalidade",
    "aggregation_notice": "Os valores apresentados são agregados..."
  }
}

Resultado conceitual do notebook

Tabelas estatísticas

        ↓

Seleção dos indicadores

        ↓

Organização por tema

        ↓

Geração de texto epidemiológico

        ↓

Inclusão de escopo e proveniência

        ↓

Documentos semânticos agregados

        ↓

JSON / JSONL / CSV

        ↓

Indexação no RAG


O JSONL será o formato mais adequado para a próxima etapa, porque cada linha poderá ser convertida diretamente em um objeto Document do LangChain ou em uma entrada independente para FAISS ou Chroma.